# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs in the dataset. All entities are referenced by their `@id` fields.

Let's list all record sets, their `@id`, the associated columns (field `@id`s), and column labels if available.

In [ ]:
# List all record sets in the dataset
record_sets = list(metadata.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id', f)} (label: {f.get('name', '')})")
        else:
            print(f"    - {f}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**TIP:** Choose the most informative record set that contains the majority of relevant experiment records (typically the primary data table).

In [ ]:
# Get record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set '{record_set_id}' with shape {df.shape}")
    except Exception as e:
        print(f"Warning: Could not load records for {record_set_id} due to: {e}")

# Choose the main record set for further analysis (use the first one with data as example)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nField columns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular record set with data was loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalization, and grouping. All selections are done using the `@id` of fields.

Below we:
- Select a numeric field (e.g., `columns: cr:MW`)
- Filter records where MW > a given threshold
- Normalize the MW column
- Optionally group by another field (e.g., `cr:Accession` or `cr:Description` if categorical) and aggregate

In [ ]:
# Example: Analyze MW (Molecular Weight) field
# Find the MW numeric field by @id (likely something like 'cr:MW')
MW_field = None
for field in dataframes[main_record_set_id].columns:
    if 'MW' in field or 'mw' in field or 'MolecularWeight' in field:
        MW_field = field
        break

if MW_field is None:
    raise ValueError('Could not find a field representing Molecular Weight (MW) in the columns.')

print(f"Using MW field: {MW_field}")

# Convert MW to numeric if not already
df = dataframes[main_record_set_id]
df[MW_field] = pd.to_numeric(df[MW_field], errors='coerce')

threshold = 100000  # Example threshold for molecular weight (~100 kDa)
filtered_df = df[df[MW_field] > threshold].copy()
print(f"Filtered records with {MW_field} > {threshold} (count: {len(filtered_df)}):")
display(filtered_df.head())

# Normalize MW
filtered_df[f"{MW_field}_normalized"] = (filtered_df[MW_field] - filtered_df[MW_field].mean()) / filtered_df[MW_field].std()
print(f"Normalized {MW_field} for filtered records:")
display(filtered_df[[MW_field, f"{MW_field}_normalized"]].head())

# Try grouping by 'Description' or similar field
group_field_candidates = [col for col in df.columns if 'Description' in col or 'description' in col or 'Accession' in col or 'accession' in col]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[MW_field].mean().reset_index().sort_values(MW_field, ascending=False)
    print(f"Grouped data by {group_field}, showing mean {MW_field}:")
    display(grouped_df.head())
else:
    print('No suitable grouping field found.')

## 5. Visualization
Visualize the distribution of the molecular weight and relationship to another field.

Below, we show a histogram of MW and a scatter plot if another numeric field (e.g., Normalized Abundance) is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[MW_field].dropna(), bins=30, kde=True)
plt.xlabel('Molecular Weight (MW)')
plt.title('Distribution of Molecular Weight')
plt.show()

# Try to plot MW vs normalized abundance if such a field exists
abundance_field = None
for f in df.columns:
    if 'abundance' in f.lower() and df[f].dtype in [int, float]:
        abundance_field = f
        break

if abundance_field:
    plt.figure(figsize=(7,5))
    sns.scatterplot(data=df, x=MW_field, y=abundance_field)
    plt.xlabel('Molecular Weight')
    plt.ylabel(abundance_field)
    plt.title(f'{abundance_field} vs. MW')
    plt.show()
else:
    print('No normalized abundance field found for scatter plotting.')

## 6. Conclusion
In this notebook, we've loaded a mass spectrometry dataset using Croissant schema and the `mlcroissant` library. We reviewed available record sets and fields by their `@id`, extracted data from the main record set, filtered and normalized a numeric field (MW), grouped records for summary statistics, and visualized the molecular weight distribution.

This explorative workflow can be adapted to other Croissant datasets with minimal changes. For in-depth analyses, consult the column field `@id` mapping to the original experimental variables and documentation provided with the dataset.